# Bench 实验
从上到下运行；标「可重跑」的格可以单独再跑。ESP 每次 reset 时保持线圈电源关闭。
结束后：`uv run python controller/control/safe_off.py`，关电源。步骤说明见 `README.md`。

In [ ]:
# 导入
from pathlib import Path
import sys
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'LLM/bench.py').exists())
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))
from IPython.display import display, Image
from LLM.bench import Bench

In [ ]:
# 设置：改任何一项都从下一格重跑
CAMERAS = (1, 0)                                   # (A, B) 相机序号
PORT = '/dev/cu.usbserial-0001'                    # ESP 串口
CALIBRATION = PROJECT/'LLM/runs/setup_002/calibration/stereo_rig.json'  # None = 重新标定
PLUMB = None   # 装置定位用的垂线像素 {'bottom': {'a': [x, y], 'b': [x, y]}, 'top': {...}}；None = sync check
RAMP = 'alignment_50Hz'                            # LLM/ramps/ 中的文件名
NOTE = ''                                          # robot / bearing / 线圈、实验目的
print('可选 ramp:', sorted(p.stem for p in (PROJECT/'LLM/ramps').glob('*.json')))

In [ ]:
# 建立 bench（新 session；之前的验证全部作废）
bench = Bench(cameras=CAMERAS, serial_port=PORT)

In [ ]:
# 相机预览（可重跑）：遮挡一个镜头确认 A/B；对焦；之后不再动镜头
bench.preview_cameras()

In [ ]:
# 标定（可重跑）：首次、重新对焦、单台移动、换模式时设 CALIBRATION = None。板子静止摆多个姿态，按 q
if CALIBRATION is None:
    bench.collect_charuco_frames()
    bench.calibrate_intrinsics()
    bench.calibrate_extrinsics()
else:
    bench.use_calibration(CALIBRATION)

In [ ]:
# 在最终机位验证标定（可重跑）：>=5 个新的静止板子姿态，按 q；须 PASS
display(bench.verify_calibration())

In [ ]:
# 参考照片（可重跑）：移走板子；LED 与垂线两机可见；点 Save 再 Close
reference = bench.photograph_reference()
for tag in 'AB':
    display(Image(filename=str(reference/f'{tag}.png'), width=800))

In [ ]:
# 装置定位（可选，可重跑；步骤见 LLM/analyse/HOLDER_MARKERS.md）。PLUMB = None 时跳过 -> sync check
if PLUMB is not None:
    display(bench.find_holder_marks())
    for tag in 'AB':
        display(Image(filename=str(reference/f'{tag}_holder_landmarks.png'), width=800))
    display(bench.locate_fixture(PLUMB))

In [ ]:
# 10 秒纯相机试录（可重跑），自动审核
display(bench.record())

In [ ]:
# 查看 ramp（可重跑）
display(bench.show_ramp(RAMP))

In [ ]:
# 实验：Record -> START（LED + 驱动）-> 自动 stop。会驱动线圈！看到 READY 后再开线圈电源
display(bench.run(RAMP, note=NOTE))

In [ ]:
# 分析（可重跑）：LED 对时 -> frequency + 四路 CS 曲线
display(bench.analyse())
for name in ('input_curves.png', 'overview.png'):
    if (bench.last_run/'export'/name).exists():
        display(Image(filename=str(bench.last_run/'export'/name)))